# **IMPORTING LIBRARIES**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

import random

# **LOADING DATASET**

In [ ]:
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)

print(df.head())

         Date      Time Global_active_power Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.840   
1  16/12/2006  17:25:00               5.360                 0.436  233.630   
2  16/12/2006  17:26:00               5.374                 0.498  233.290   
3  16/12/2006  17:27:00               5.388                 0.502  233.740   
4  16/12/2006  17:28:00               3.666                 0.528  235.680   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0           18.400          0.000          1.000            17.0  
1           23.000          0.000          1.000            16.0  
2           23.000          0.000          2.000            17.0  
3           23.000          0.000          1.000            17.0  
4           15.800          0.000          1.000            17.0  


# **DATA CLEANING & TYPE CONVERSION**

In [ ]:
df.replace("?", np.nan, inplace=True)

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df[numeric_cols] = df[numeric_cols].astype(float)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# **FEATURE ENGINEERING – SUB-METERING**

In [ ]:
df["total_sub_metering"] = (
    df["Sub_metering_1"] +
    df["Sub_metering_2"] +
    df["Sub_metering_3"]
)

# **FUNCTION FOR IDENTIFYING HEAVILY USED SUB-METERING**

In [ ]:
def get_heavy_submeter(row):
    subs = {
        "Sub_metering_1": row["Sub_metering_1"],
        "Sub_metering_2": row["Sub_metering_2"],
        "Sub_metering_3": row["Sub_metering_3"]
    }
    return max(subs, key=subs.get)

df["heavy_submeter"] = df.apply(get_heavy_submeter, axis=1)

# **LOAD STATE DEFINITION**

In [ ]:
def get_load_state(row):
    if row["Global_active_power"] > 4.0:
        return "HEAVY"
    elif row["Global_active_power"] > 2.5:
        return "MEDIUM"
    else:
        return "LOW"

df["load_state"] = df.apply(get_load_state, axis=1)


# **CLUSTERING (USAGE PATTERN DISCOVERY)**

In [ ]:
cluster_features = df[[
    "Global_active_power",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]]

scaler = MinMaxScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=3, random_state=42)
df["usage_cluster"] = kmeans.fit_predict(cluster_scaled)

# **LSTM – LOAD PREDICTION PREPARATION**

In [ ]:
power_data = df[["Global_active_power"]].values

scaler_lstm = MinMaxScaler()
power_scaled = scaler_lstm.fit_transform(power_data)

def create_sequences(data, seq_len=24):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(power_scaled)


# **LSTM MODEL**

In [ ]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)),
    LSTM(50),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.fit(X, y, epochs=5, batch_size=64, verbose=1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
32020/32020 ━━━━━━━━━━━━━━━━━━━━ 231s 7ms/step - loss: 6.7110e-04
Epoch 2/5
32020/32020 ━━━━━━━━━━━━━━━━━━━━ 226s 7ms/step - loss: 5.5223e-04
Epoch 3/5
32020/32020 ━━━━━━━━━━━━━━━━━━━━ 264s 7ms/step - loss: 5.2001e-04
Epoch 4/5
32020/32020 ━━━━━━━━━━━━━━━━━━━━ 262s 7ms/step - loss: 4.9392e-04
Epoch 5/5
32020/32020 ━━━━━━━━━━━━━━━━━━━━ 268s 7ms/step - loss: 4.8193e-04


# **LSTM LOAD PREDICTION**

In [ ]:
df["predicted_load"] = np.nan

predictions = model.predict(X, verbose=0)
predictions = scaler_lstm.inverse_transform(predictions)

df.loc[df.index[-len(predictions):], "predicted_load"] = predictions.flatten()


# **Q-LEARNING SETUP**

In [ ]:
states = ["LOW", "MEDIUM", "HEAVY"]
actions = ["NO_ACTION", "SHIFT", "REDUCE_AND_SHIFT"]

Q = np.zeros((len(states), len(actions)))

alpha = 0.1
gamma = 0.9
epsilon = 0.1


# **REWARD FUNCTION (SUB-METERING AWARE)**

In [ ]:
def get_reward(state, action):
    if state == "HEAVY" and action == "REDUCE_AND_SHIFT":
        return 10
    if state == "HEAVY" and action in ["SHIFT"]:
        return -2
    if state == "LOW" and action == "NO_ACTION":
        return 5
    return -1


# **Q-LEARNING TRAINING LOOP**

In [ ]:
for i in range(len(df)):
    state = df.loc[i, "load_state"]
    state_idx = states.index(state)

    if random.random() < epsilon:
        action_idx = random.randint(0, len(actions)-1)
    else:
        action_idx = np.argmax(Q[state_idx])

    action = actions[action_idx]
    reward = get_reward(state, action)

    Q[state_idx, action_idx] += alpha * (
        reward + gamma * np.max(Q[state_idx]) - Q[state_idx, action_idx]
    )


# **FINAL ACTION DECISION**

In [ ]:
def decide_action(row):
    state_idx = states.index(row["load_state"])
    return actions[np.argmax(Q[state_idx])]

df["final_action"] = df.apply(decide_action, axis=1)


# **APPLIANCE MAPPING**

In [ ]:
submeter_appliance_map = {
    "Sub_metering_1": "Kitchen appliances",
    "Sub_metering_2": "Laundry appliances",
    "Sub_metering_3": "Heater / AC / Water heater"
}


# **FINAL RECOMMENDATION ENGINE**

In [ ]:
def generate_recommendation(row):
    appliance = submeter_appliance_map[row["heavy_submeter"]]

    if row["final_action"] == "REDUCE_AND_SHIFT":
        return f"Reduce usage and shift {appliance} to off-peak / solar hours"
    elif row["final_action"] == "SHIFT":
        return f"Shift {appliance} to off-peak / solar hours"
    else:
        return "No action required"

df["recommendation"] = df.apply(generate_recommendation, axis=1)


# **FINAL OUTPUT**

In [ ]:
df[[
    "Global_active_power",
    "predicted_load",
    "usage_cluster",
    "heavy_submeter",
    "load_state",
    "final_action",
    "recommendation"
]].head(10)


,Global_active_power,predicted_load,usage_cluster,heavy_submeter,load_state,final_action,recommendation
0,4.216,NaN,1,Sub_metering_3,HEAVY,REDUCE_AND_SHIFT,Reduce usage and shift Heater / AC / Water hea...
1,5.360,NaN,1,Sub_metering_3,HEAVY,REDUCE_AND_SHIFT,Reduce usage and shift Heater / AC / Water hea...
2,5.374,NaN,1,Sub_metering_3,HEAVY,REDUCE_AND_SHIFT,Reduce usage and shift Heater / AC / Water hea...
3,5.388,NaN,1,Sub_metering_3,HEAVY,REDUCE_AND_SHIFT,Reduce usage and shift Heater / AC / Water hea...
4,3.666,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
5,3.520,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
6,3.702,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
7,3.700,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
8,3.668,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
9,3.662,NaN,1,Sub_metering_3,MEDIUM,NO_ACTION,No action required
